# Блокнот для обучения Qwen3 VL 4B 
**Большая часть кодов была взята из официального Colab-блокнота Unsloth, посвященного обучению Qwen3 VL 8B**

***Daniel Han, Michael Han and Unsloth team,
http://github.com/unslothai/unsloth,
2023***

## Принцип последовательного обучения без забывания

При обучении LoRA поверх старого адаптера, он перезаписывается — модель «забывает» то, что она только что выучила.

После каждой стадии необходимо **сливать LoRA в базу** и сохранять **полный чекпоинт (merged model)**. На следующей стадии загружать этот merged model как новую базу и вешать **новый, пустой** LoRA — так знания text остаются в базе, а LoRA учит только math (и так далее для vision).

1. text: база = `unsloth/Qwen3-VL-8B-...` → обучаем LoRA → сливаем в базу → сохраняем в `merged_after_text`.
2. math: база = `merged_after_text` → новый LoRA → обучаем → сливаем → сохраняем в `merged_after_math`.
3. vision: база = `merged_after_math` → новый LoRA → обучаем → сливаем → сохраняем финальный результат.

В связи с этим для полного обучения на всех трех стадиях необходимо несколько раз менять значения в коде и перезапускать его.

## Установка библиотек
А также ранняя инициализация unsloth для предотвращения ошибок

In [ ]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Only in Colab notebooks
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==3.6.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install unsloth
!pip install transformers==4.57.1
!pip install trl==0.22.2
!pip install "datasets[zstd]" zstandard
!pip install jsonlines

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs

## Распаковка merged модели из предыдущей стадии
*Требуется только для math и vision*

In [ ]:
from google.colab import drive
import zipfile
drive.mount("/content/drive")

In [ ]:
with zipfile.ZipFile('/content/drive/MyDrive/after-math/after-math.zip', 'r') as z: # .../after-math/after-math.zip - архив в папке НА ДИСКЕ
  z.extractall("/content/after-math")

## Аргументы для загрузки датасетов \ модели, обучения

In [ ]:
from typing import Dict, List, Optional, Any
import os

class Args():
    outdir = "/kaggle/working/"
    val = 0.02

    # Путь к базовой модели для текущей стадии
    # Для text — оригинальная unsloth-модель; для math/vision — merged после предыдущей стадии
    ORIGINAL_BASE = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"
    merged_after_text = "/content/after-text"   # путь после обучения text, напр. os.path.join(outdir, "merged_after_text")
    merged_after_math = "/content/after-math"  # путь после обучения math, напр. os.path.join(outdir, "merged_after_math")

    # Лимиты
    limit_text = 20000
    limit_math = 20000
    limit_vision = 10000
    max_length = 2048

    # Эпохи обучения
    epochs_text = 1.0
    epochs_math = 1.0
    epochs_vision = 1.0

    # Размеры батча
    batch_train = 4
    batch_eval = 4
    grad_accum = 4

    lr = 2e-5 # 1e-4 стоит попробовать
    weight_decay = 0.01
    num_workers = 2

    # МЕНЯТЬ СТАДИЮ
    stage = "text" # "text" | "math" | "vision"

    @classmethod
    def get_base_model_path(cls) -> str:
        """Путь к модели, с которой начинаем текущую стадию (база + уже слитые предыдущие стадии)."""
        if cls.stage == "text":
            return cls.ORIGINAL_BASE
        if cls.stage == "math":
            path = cls.merged_after_text or os.path.join(cls.outdir, "merged_after_text")
            if not os.path.isfile(os.path.join(path, "config.json")):
                raise FileNotFoundError(f"Сначала обучите стадию 'text' и сохраните merged в: {path}")
            return path
        if cls.stage == "vision":
            path = cls.merged_after_math or os.path.join(cls.outdir, "merged_after_math")
            if not os.path.isfile(os.path.join(path, "config.json")):
                raise FileNotFoundError(f"Сначала обучите стадию 'math' и сохраните merged в: {path}")
            return path
        raise ValueError(f"Unknown stage: {cls.stage}")

args = Args()

## Оптимизация
*Запуск не обязателен, но желателен*

In [ ]:
import os
import torch

# Внимание: T4 GPUs (Turing архитектура) не поддерживает TF32 (только Ampere+)
torch.backends.cuda.matmul.allow_tf32 = False  # Но не повредит
torch.backends.cudnn.allow_tf32 = False

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"  # Асинхронное выполнение

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Не выводит ошибки токенизатора

os.environ["DS_SKIP_CUDA_CHECK"] = "0"  # Проверка CUDA на всякий

torch.backends.cudnn.benchmark = True  # Optimization
torch.backends.cudnn.deterministic = False

torch.cuda.empty_cache()  # Очистка кэша перед обучением

print("Доп. Оптимизация")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"  CUDA Доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## Подготовка данных
*Можно сказать это самый сложный этап — так как требует именно самостоятельно придуманного кода*

### Функции загрузки датасетов

In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset

def load_bactrian_x(limit: Optional[int], num_workers) -> List:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("MBZUAI/Bactrian-X", "ru", num_proc=num_workers, split=split_expr)
    def _convert(sample):
        problems = f"{sample['instruction']} — {sample['input']}"
        solutions = sample["output"]
        conversations = []
        for problem, solution in zip(problems, solutions):
            conversations.append([
                {"role" : "user",      "content" : problem},
                {"role" : "assistant", "content" : solution}
            ])
        return {"conversations" : conversations}
    converted_ds = tokenizer.apply_chat_template(list(ds.map(_convert, batched=True)["conversations"]), tokenize = False, add_generation_prompt=False)
    return converted_ds

def load_ru_turbo_alpaca(limit: Optional[int], num_workers) -> List:
    ds = load_dataset("IlyaGusev/ru_turbo_alpaca", num_proc=num_workers, trust_remote_code=True)
    if limit:
        ds_train = ds["train"].select(range(min(limit, len(ds["train"]))))
    def _convert(sample):
        problems  = f"{sample['instruction']} — {sample['input']}"
        solutions = sample["output"]
        conversations = []
        for problem, solution in zip(problems, solutions):
            conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
            ])
        return { "conversations": conversations, }
    converted_ds = tokenizer.apply_chat_template(list(ds_train.map(_convert, batched = True)["conversations"]), tokenize = False, add_generation_prompt=False)
    return converted_ds # type: ignore

def build_text_stage(limit: Optional[int], num_workers) -> List:
    parts = []
    for loader in (load_bactrian_x, load_ru_turbo_alpaca):
        try:
            parts.append(loader(limit=limit, num_workers=num_workers))
        except Exception as e:
            print(f"При загрузке {loader} возникла ошибка: {e}")
    if not parts:
        raise RuntimeError("No text datasets loaded.")
    try:
        ds = [*parts[0], *parts[1]]
    except Exception as e:
        raise RuntimeError(f"При загрузке датасетов для стадии {args.stage} возникла ошибка: {e}")
    return ds


# -------------- Loaders: MATH (TEXT) --------------

def load_ru_alpaca_math(limit: Optional[int], num_workers) -> List:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("ai-bond/ru-alpaca-math", split=split_expr, num_proc=num_workers)
    def _convert(sample):
        problems  = f"{sample['instruction']} — {sample['input']}"
        solutions = sample["output"]
        conversations = []
        for problem, solution in zip(problems, solutions):
            conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
            ])
        return { "conversations": conversations, }
    converted_ds = tokenizer.apply_chat_template(list(ds.map(_convert, batched = True)["conversations"]), tokenize = False, add_generation_prompt=False)
    return converted_ds # type: ignore

def load_gsm8k_ru(limit: Optional[int], num_workers) -> List:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("d0rj/gsm8k-ru", split=split_expr, num_proc=num_workers)
    def _convert(sample):
        problems  = sample['question']
        solutions = sample["answer"]
        conversations = []
        for problem, solution in zip(problems, solutions):
            conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
            ])
        return { "conversations": conversations, }
    converted_ds = tokenizer.apply_chat_template(list(ds.map(_convert, batched = True)["conversations"]), tokenize = False, add_generation_prompt=False)
    return converted_ds # type: ignore

def build_math_stage(limit: Optional[int], num_workers) -> List:
    parts = []
    for loader in (load_ru_alpaca_math, load_gsm8k_ru):
        try:
            parts.append(loader(limit=limit, num_workers=num_workers))
        except Exception as e:
            print(f"При загрузке {loader} возникла ошибка: {e}")
    if not parts:
        raise RuntimeError("No math datasets loaded.")
    try:
        ds = [*parts[0], *parts[1]]
    except Exception as e:
        raise RuntimeError(f"При загрузке датасетов для стадии {args.stage} возникла ошибка: {e}")
    return ds


# -------------- Loaders: VISION (MATH IMAGES) --------------

def load_chartqa(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("HuggingFaceM4/ChartQA", split=split_expr, num_proc=num_workers)
    def _convert(batch):
        conversations = []
        for q, img, labels in zip(batch["query"], batch["image"], batch["label"]):
            conversations.append([
                    {
                    "role": "user",
                    "content":  [
                        {"type": "text",  "text": q},
                        {"type": "image", "image": img},
                                ],
                    },
                    {
                    "role": "assistant",
                    "content":  [
                        {"type": "text", "text": labels[0]},
                                ],
                    },
            ]
            )
        return {"conversations": conversations}

    converted_ds = ds.map(
        _convert,
        batched=True,
        remove_columns=ds.column_names,
        num_proc=num_workers,
    )
    return converted_ds

def load_aida_calculus(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "train" if not limit else f"train[:{limit}]"
    ds = load_dataset("deepcopy/Aida-Calculus-Math-Handwriting", split=split_expr, num_proc=num_workers)
    def _convert(batch):
        instruction = "Write the LaTeX representation for this image."
        conversations = []
        for img, latex in zip(batch["image"], batch["latex"]):
            conversations.append([
                        {
                        "role": "user",
                        "content":  [
                            {"type": "text",  "text": instruction},
                            {"type": "image", "image": img},
                                    ],
                        },
                        {
                        "role": "assistant",
                        "content":  [
                            {"type": "text", "text": latex},
                                    ],
                        },
            ]
            )
        return {"conversations": conversations}

    converted_ds = ds.map(
        _convert,
        batched=True,
        remove_columns=ds.column_names,
        num_proc=num_workers,
    )
    return converted_ds

def load_geometry3k(limit: Optional[int], num_workers) -> Dataset:
    split_expr = "test" if not limit else f"test[:{limit}]"
    ds = load_dataset("hiyouga/geometry3k", split=split_expr, num_proc=num_workers)
    def _convert(batch):
        conversations = []
        for problem, images, answer in zip(batch["problem"], batch["images"], batch["answer"]):
            img = images[0] if isinstance(images, list) else images
            conversations.append([
                        {
                        "role": "user",
                        "content":  [
                            {"type": "text",  "text": problem},
                            {"type": "image", "image": img},
                                    ],
                        },
                        {
                        "role": "assistant",
                        "content":  [
                            {"type": "text", "text": answer},
                                    ],
                        },
            ]
            )
        return {"conversations": conversations}

    converted_ds = ds.map(
        _convert,
        batched=True,
        remove_columns=ds.column_names,
        num_proc=num_workers,
    )
    return converted_ds

def build_vision_stage(limit: Optional[int], num_workers) -> Dataset:
    parts = []
    for loader in (load_chartqa, load_aida_calculus, load_geometry3k):
        try:
            parts.append(loader(limit=limit, num_workers=num_workers))
        except Exception as e:
            print(f"При загрузке {loader} возникла ошибка: {e}")
    if not parts:
        raise RuntimeError("No vision datasets loaded.")
    return concatenate_datasets(parts)

## Загрузка датасетов, модели и обучение

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl.trainer.sft_trainer import SFTTrainer # from trl
from trl.trainer.sft_config import SFTConfig # from trl

# Determine epochs based on stage
epochs_map = {
    "text": args.epochs_text,
    "math": args.epochs_math,
    "vision": args.epochs_vision
}
current_epochs = epochs_map.get(args.stage, args.epochs_text)

training_args = SFTConfig(
    per_device_train_batch_size=args.batch_train,
    per_device_eval_batch_size=args.batch_eval,
    gradient_accumulation_steps=args.grad_accum,
    fp16=True,
    fp16_full_eval=True,
    warmup_ratio=0.03,  # 3% warmup
    num_train_epochs=current_epochs,
    save_strategy="steps",
    save_steps=50,
    eval_steps=10,
    eval_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=args.lr,
    logging_steps=10,  # период высчитывания training loss
    bf16=False,
    optim="adamw_torch",
    weight_decay=args.weight_decay,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir=args.outdir,
    report_to="none",

    # Performance optimizations
    dataloader_pin_memory=True,
    dataloader_num_workers=args.num_workers,


    # ОБЯЗАТЕЛЬНО для обучения на изображениях:
    remove_unused_columns=False if args.stage == "vision" else True,
    dataset_text_field="" if args.stage == "vision" else "text",
    dataset_kwargs={"skip_prepare_dataset": True} if args.stage == "vision" else None,
    max_length=args.max_length,

    # AttributeError: 'int' object has no attribute 'mean'
    # This happens with DeepSpeed when token counting returns integers instead of tensors
    # Setting to False fixes the error.
    average_tokens_across_devices=False,
)



In [ ]:
import torch

# База: для text — оригинал; для math/vision — merged после предыдущей стадии (новый LoRA каждый раз).
base_path = args.get_base_model_path()
print(f"Загрузка базы для стадии '{args.stage}': {base_path}")

model, tokenizer = FastVisionModel.from_pretrained(
    base_path,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True if args.stage == "vision" else False, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # Unsloth supports rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
import pandas as pd

if args.stage == "text": # ЕСЛИ СТАДИЯ == text
    ds = build_text_stage(limit=args.limit_text, num_workers=args.num_workers) # -> List
    ds = pd.Series(ds) # -> pd.Series
    ds.name = "text" # добавляет ключ
    ds = Dataset.from_pandas(pd.DataFrame(ds)) # -> Dataset
    ds = ds.shuffle(seed = 42)
    ds = ds.train_test_split(test_size=args.val, seed=42) # -> DatasetDict
    ds_train = ds["train"]
    ds_eval  = ds["test"]
elif args.stage == "math": # ЕСЛИ СТАДИЯ == math
    ds = build_math_stage(limit=args.limit_math, num_workers=args.num_workers) # -> List
    ds = pd.Series(ds) # -> pd.Series
    ds.name = "text" # добавляет ключ
    ds = Dataset.from_pandas(pd.DataFrame(ds)) # -> Dataset
    ds = ds.shuffle(seed = 42)
    ds = ds.train_test_split(test_size=args.val, seed=42) # -> DatasetDict
    ds_train = ds["train"]
    ds_eval  = ds["test"]
else: # ЕСЛИ СТАДИЯ == vision
    ds = build_vision_stage(limit=args.limit_vision, num_workers=args.num_workers) # -> Dataset
    ds = ds.train_test_split(test_size=args.val, seed=42) # -> DatasetDict
    ds_train = ds["train"]
    ds_eval  = ds["test"]

In [ ]:
FastVisionModel.for_training(model) # Переключение на обучение

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer) if args.stage == "vision" else None,
    train_dataset = ds_train,
    eval_dataset = ds_eval,
    args = training_args,
)

from transformers import EarlyStoppingCallback
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience = 3,     # Сколько ПРОВЕРОК нужно ждать перед тем как остановить обучение (при этом loss не улучшается)
    early_stopping_threshold = 0.007,  # Устанавливает, на сколько максимум может изменится loss но обучение
                                        # все равно остановится. Например, если значение = 0.1, при этом loss был 0.02
                                        # и стал 0.01 в промежутке <early_stopping_patience>, то обучение останавливается
)
trainer.add_callback(early_stopping_callback)

In [ ]:
trainer_stats = trainer.train()

## Сохранение модели
Два варианта — в адаптерах и GGUF

### Merged checkpoint для следующей стадии (обязательно)

Сливаем LoRA в базу и сохраняем полный чекпоинт. Его загружаем как базу на следующей стадии (см. Args).

In [ ]:
# Слить LoRA в базу и сохранить полную модель для следующей стадии
merged_dir = os.path.join(args.outdir, f"merged_after_{args.stage}")
model.save_pretrained_merged(
    merged_dir,
    tokenizer,
    save_method="merged_16bit",
)

### Запаковка в архив для скачивания

In [ ]:
!zip -r "after_{args.stage}.zip" "/kaggle/working/merged_after_{args.stage}"
from IPython.display import FileLink
FileLink(...) # здесь должен быть путь, бесполезно в google colab